In [1]:
import pandas as pd
import numpy as np

In [37]:
# Загружаем датасеты из двух источников

hh=pd.read_csv('df_hh_rich.csv')
sj=pd.read_csv('df_sj_api.csv')

In [38]:
hh.head(1)

,title_x,address,company,short_description,experience,salary,region__title,country_title,client_staff_count,salary_num
0,Разработчик,Москва,"Русская Медиагруппа, радиохолдинг",Опыт работы,Опыт 3-6 лет,NaN,Московская область,Россия,NaN,NaN


In [39]:
sj.head(1)

,profession,currency,vacancyRichText,education,experience,firm_name,town_name,region__title,country_title,client_staff_count,position,salary
0,Дежурный специалист по информационной безопасн...,rub,"ФФКУ ""Налог-Сервис"" ФНС России по ЦОД- это пр...",Не имеет значения,От 1 года,Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,Уфа,Республика Башкортостан,Россия,100 — 500,"['Другое', 'Информационная безопасность']",0.0


In [40]:
sj['currency'].unique()

<StringArray>
['rub', 'kzt']
Length: 2, dtype: str

In [41]:
sj[sj['currency']=='kzt']

,profession,currency,vacancyRichText,education,experience,firm_name,town_name,region__title,country_title,client_staff_count,position,salary
264,Senior Kotlin разработчик,kzt,"Привет! Мы — Антара , аккредитованная IT-комп...",Не имеет значения,От 3 лет,Антара,Алматы,NaN,Казахстан,100 — 500,"['Разработка, программирование']",1600000.0


In [42]:
# вручную переведем по текущему курсу из тенге в рубли

sj.loc[264, 'salary'] = 256640.0

Теперь все вакансии в рублях, как и у хх.ру, поэтому можно удалить столбец currency из  sj.

Удалим также столбцы town_name в sj и address, тк мы специально добавляли регионы и страны, чтобы по ним анализировать в едином виде.

Удалим столбец salary из hh, тк мы преобразовали его в более полезный числовой salary_num и position, тк он есть только в sj и не является ценным для гипотез в таком небольшом объеме (в отличие от education - требования к образованию).

In [43]:
sj=sj.drop(columns=['currency','town_name','position'])
hh=hh.drop(columns=['address','salary'])

# Приводим названия признаков к единому виду

In [45]:
hh=hh.rename(columns={
    'title_x': 'profession',
    'salary_num': 'salary',
    'short_description':'vac_description',
    'region__title':'region',
    'country_title':'country',
    'client_staff_count':'company_staff_count',
})

sj=sj.rename(columns={
    'vacancyRichText':'vac_description',
    'firm_name':'company',
    'region__title':'region',
    'country_title':'country',
    'client_staff_count':'company_staff_count',
})

# Соединяем датасеты

In [46]:
final_df = pd.concat([hh, sj], ignore_index=True)
final_df.head()

,profession,company,vac_description,experience,region,country,company_staff_count,salary,education
0,Разработчик,"Русская Медиагруппа, радиохолдинг",Опыт работы,Опыт 3-6 лет,Московская область,Россия,NaN,NaN,NaN
1,Разработчик,Детский хоспис Дом с маяком,"Уверенное знание Python 3, базовых принципов О...",Опыт 1-3 года,Московская область,Россия,NaN,95000.0,NaN
2,Backend-разработчик,ООО ГК СтиС,2+ года коммерческой разработки на PHP 7.4+. У...,Опыт 1-3 года,Московская область,Россия,NaN,NaN,NaN
3,Разработчик ЦФТ,Bell Integrator,Опыт работы с АБС «ЦФТ-Банк» в качестве,Опыт 3-6 лет,Московская область,Россия,NaN,NaN,NaN
4,Разработчик C++,ООО БУЛАТ,Опыт разработки или исправления/доработки внут...,Опыт 3-6 лет,Московская область,Россия,NaN,NaN,NaN


# Приводим к единому виду признак требуемого опыта

In [47]:
final_df['experience'].unique()

<StringArray>
[     'Опыт 3-6 лет',     'Опыт 1-3 года',  'Опыт более 6 лет',
         'Без опыта',         'От 1 года',          'От 3 лет',
 'Не имеет значения',          'От 6 лет']
Length: 8, dtype: str

Преобразуем значения в числовое - оставим минимальный требуемый опыт. 'Не имеет значения'=0. 

In [49]:
final_df['experience'] = final_df['experience'].replace({
    'Опыт 1-3 года': 1,
    'От 1 года': 1,
    
    'Опыт 3-6 лет': 3,
    'От 3 лет': 3,
    
    'Опыт более 6 лет': 6,
    'От 6 лет': 6,
    
    'Без опыта': 0,
    'Не имеет значения': 0
})
final_df.head()

,profession,company,vac_description,experience,region,country,company_staff_count,salary,education
0,Разработчик,"Русская Медиагруппа, радиохолдинг",Опыт работы,3,Московская область,Россия,NaN,NaN,NaN
1,Разработчик,Детский хоспис Дом с маяком,"Уверенное знание Python 3, базовых принципов О...",1,Московская область,Россия,NaN,95000.0,NaN
2,Backend-разработчик,ООО ГК СтиС,2+ года коммерческой разработки на PHP 7.4+. У...,1,Московская область,Россия,NaN,NaN,NaN
3,Разработчик ЦФТ,Bell Integrator,Опыт работы с АБС «ЦФТ-Банк» в качестве,3,Московская область,Россия,NaN,NaN,NaN
4,Разработчик C++,ООО БУЛАТ,Опыт разработки или исправления/доработки внут...,3,Московская область,Россия,NaN,NaN,NaN


# Скачиваем тоговый датасет

In [50]:
final_df.to_csv('final_df.csv', index=False)

# Анализ признаков

In [51]:
final_df.isna().sum()/len(final_df)

profession             0.000000
company                0.000000
vac_description        0.008603
experience             0.000000
region                 0.135909
country                0.051700
company_staff_count    0.955166
salary                 0.727852
education              0.958226
dtype: float64